# SNR-PER-016 의사결정 추론 처리시간 시험

- **시험 대상:** 주차장별 48시간 잔여 주차면수 예측 서비스 (`GET /api/tci/ai_engine/park_pred`)
- **성능 기준:** 의사결정 추론 처리시간 **2초 이하** (5차년도 목표, 3차년도 목표 4초)
- **시험 규모:** 주차장 10개 × 30회 = 요청 300회, 전체가 2초 이하여야 합격
- **모델 구성:** 주차장 10개, `RandomForestRegressor`(트리 1,000개, `random_state=42`)
- **실행 방법:** Run → Run All Cells (실행 순서: 데이터 내려받기 → 전처리 → 모델 학습 또는 재사용 → 서버 구동 → 반복 측정 → 결과 저장)
- **검증 항목:** 48개 예측값, 1시간 간격, 처리시간, 순차 호출과 일괄 호출의 예측값 일치, 오류 응답(404·400)
- **서버 측정 구간:** 요청 처리 시작부터 모델 로딩·특징 생성·추론·결과 객체 생성까지
- **클라이언트 측정 구간:** HTTP 요청 직전부터 응답 본문 전체 수신까지
- **통신 환경:** 같은 컨테이너의 loopback HTTP
- **데이터 출처:** Daniel Stolfi, *Parking Birmingham*, UCI Machine Learning Repository, DOI 10.24432/C51K5Z, CC BY 4.0
- **데이터 내려받기 경로:** https://archive.ics.uci.edu/static/public/482/parking%2Bbirmingham.zip (자료 페이지: https://archive.ics.uci.edu/dataset/482/parking+birmingham)

**폴더 구성** (이 노트북은 `notebooks/`에 있습니다)

| 폴더 | 내용 |
|---|---|
| `requirements.txt` | 시험에 사용한 패키지 버전 |
| `data/` | 데이터셋 정보(`DATASET_MANIFEST.json`: 내려받기 주소, 해시). 내려받은 파일은 이곳에 저장되며 git에는 올리지 않습니다. |
| `models/` | 학습된 주차장 모델 10개(`parking_model_<ID>.pkl`), `MODEL_MANIFEST.json`(해시·학습 환경), `training.csv`(학습 결과) |
| `results/` | 시험 결과: `<실행 시각>/` 폴더(측정표·요약·그래프·응답 원문), 실행된 노트북 |

## 1. 설정

시험 폴더(`snr_per_016`)를 찾고 반복 횟수와 기준값을 정합니다. 모델은 `models/`에 있고 데이터와 학습 환경이 같으면 **재사용**합니다(`RETRAIN = False`). 다르거나 `RETRAIN = True`이면 처음부터 학습합니다.

In [ ]:
from pathlib import Path
here = Path.cwd().resolve()
PACKAGE = next((folder for folder in [here, *here.parents] if (folder / 'data/DATASET_MANIFEST.json').is_file() and (folder / 'notebooks').is_dir()), None)
assert PACKAGE is not None, 'snr_per_016 폴더 안에서 노트북을 여십시오'
MODEL_DIR = PACKAGE / 'models'      # 학습 모델 저장 폴더
DATA_DIR = PACKAGE / 'data'         # 데이터셋 저장 폴더
OUTPUT_ROOT = PACKAGE / 'results'
REPEATS = 30  # 주차장별 반복 횟수, 기본 총 300회
BASELINE_REPEATS = 1  # 개별 예측 비교 횟수, 주차장별 최소 1회
THRESHOLD_SECONDS = 2.0
INSTALL_MISSING = True
RETRAIN = False  # True: 저장된 모델이 있어도 다시 학습
print('시험 폴더:', PACKAGE)

## 2. 필요한 패키지 확인

없는 패키지만 설치합니다(`INSTALL_MISSING = True`). 설치된 버전은 뒤에서 `manifest.json`에 기록됩니다.

저장된 모델은 `requirements.txt`의 버전(Python 3.11, scikit-learn 1.7.2, numpy 2.4.6)에서 학습되었습니다. 같은 버전의 환경에서 실행하면 저장된 모델을 그대로 재사용하고, scikit-learn 버전이 다르면 4단계에서 자동으로 다시 학습합니다(시험 방법은 같습니다).

In [ ]:
import sys, subprocess, importlib.util
required = {'numpy':'numpy', 'pandas':'pandas', 'sklearn':'scikit-learn',
            'joblib':'joblib', 'holidays':'holidays', 'flask':'flask',
            'requests':'requests', 'matplotlib':'matplotlib', 'IPython':'ipython'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError('필수 패키지 미설치: ' + ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('실행 Python:', sys.executable)

## 3. 데이터 내려받기와 전처리

1. UCI 저장소에서 `parking_birmingham.zip`을 내려받아 `data/`에 저장합니다. 같은 SHA-256의 파일이 이미 있으면 다시 받지 않습니다(`data/DATASET_MANIFEST.json`에 기록된 값 기준).
2. 열 이름의 공백을 제거하고 `LastUpdated`를 시각으로, `Capacity`·`Occupancy`를 숫자로 변환합니다.
3. 결측, `Capacity ≤ 0`, `Occupancy < 0`, `Occupancy > Capacity`인 행을 제거하고, 같은 주차장·같은 시각의 중복을 제거합니다.
4. 기록이 100건 이상인 주차장 가운데 **기록 수가 많은 10곳**을 선택합니다(`IDS`). 주차장별 최근 `Capacity`를 `CAPACITY`로 보관합니다(혼잡도 계산에 사용).
5. 내려받은 파일의 해시, 전처리 전후 행 수, 기간, 선택한 주차장을 `DATA_PROVENANCE`로 기록합니다.

In [ ]:
import json, hashlib, platform, importlib.metadata, math, time, threading, io, zipfile
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from time import perf_counter_ns
import numpy as np
import pandas as pd
import joblib, holidays, requests
import matplotlib.pyplot as plt
from flask import Flask, request, jsonify
from werkzeug.serving import make_server
from werkzeug.serving import WSGIRequestHandler
from IPython.display import display, HTML, FileLink
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

DATA_INFO = json.loads((DATA_DIR / 'DATASET_MANIFEST.json').read_text(encoding='utf-8'))
DATA_URL = DATA_INFO['source_url']
DATA_DIR.mkdir(exist_ok=True)
archive_path = DATA_DIR / 'parking_birmingham.zip'
if archive_path.is_file() and hashlib.sha256(archive_path.read_bytes()).hexdigest() == DATA_INFO['archive_sha256']:
    content = archive_path.read_bytes()
    data_source = 'local copy (SHA-256 verified)'
else:
    r = requests.get(DATA_URL, timeout=90); r.raise_for_status()
    content = r.content
    archive_path.write_bytes(content)
    data_source = 'downloaded'
if hashlib.sha256(content).hexdigest() != DATA_INFO['archive_sha256']:
    print('경고: 내려받은 파일의 SHA-256이 DATASET_MANIFEST.json과 다릅니다. 제공처가 데이터를 갱신했을 수 있습니다.')
z = zipfile.ZipFile(io.BytesIO(content))
name = next(x for x in z.namelist() if x.endswith('.csv'))
raw = z.read(name); (DATA_DIR / 'dataset.csv').write_bytes(raw)
data = pd.read_csv(io.BytesIO(raw))
data.columns = data.columns.str.strip()
data['LastUpdated'] = pd.to_datetime(data['LastUpdated'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
for c in ['Capacity', 'Occupancy']: data[c] = pd.to_numeric(data[c], errors='coerce')
original_rows = len(data)
data = data.dropna(subset=['SystemCodeNumber', 'LastUpdated', 'Capacity', 'Occupancy'])
data = data[(data.Capacity > 0) & (data.Occupancy >= 0) & (data.Occupancy <= data.Capacity)]
data = data.drop_duplicates(['SystemCodeNumber', 'LastUpdated']).sort_values(['SystemCodeNumber', 'LastUpdated'])
counts = data.groupby('SystemCodeNumber').size().sort_values(ascending=False, kind='stable')
IDS = sorted(counts[counts >= 100].head(10).index.astype(str).tolist())
assert len(IDS) == 10
CAPACITY = {ident: float(data[data.SystemCodeNumber == ident].sort_values('LastUpdated').Capacity.iloc[-1]) for ident in IDS}
DATA_PROVENANCE = dict(url=DATA_URL, doi=DATA_INFO['doi'], license=DATA_INFO['license'], author=DATA_INFO['author'],
                       archive_sha256=hashlib.sha256(content).hexdigest(), csv_sha256=hashlib.sha256(raw).hexdigest(),
                       downloaded_at=datetime.now(ZoneInfo('Asia/Seoul')).isoformat(), source=data_source,
                       original_rows=original_rows, clean_rows=len(data),
                       start=str(data.LastUpdated.min()), end=str(data.LastUpdated.max()), selected_ids=IDS)
print('DATA', json.dumps(DATA_PROVENANCE, ensure_ascii=False), flush=True)

## 4. 모델 학습 또는 재사용

주차장마다 `RandomForestRegressor` 1개를 학습합니다.

- **입력 특징 3개:** 하루 중 분(`minutes_of_day`), 요일(`day_of_week`), 평일·비공휴일 여부(`day_type`, 영국 공휴일 기준)
- **목표값:** 잔여 주차면수 = `Capacity − Occupancy`
- **분할:** 시간 순서로 앞 80%를 학습, 뒤 20%를 검증(홀드아웃 MAE 기록)
- **모델:** 트리 1,000개, `random_state=42`, `n_jobs=1`

`models/MODEL_MANIFEST.json`에 기록된 **데이터 해시·scikit-learn 버전·주차장 목록·모델 파일 해시가 모두 현재와 같으면** 저장된 모델을 재사용해 학습을 건너뜁니다. 하나라도 다르면 학습합니다. 학습한 경우 모델 파일, `models/training.csv`, `models/MODEL_MANIFEST.json`을 새로 저장합니다.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_MANIFEST = MODEL_DIR / 'MODEL_MANIFEST.json'
def file_sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def saved_models_usable():
    if RETRAIN: print('RETRAIN=True: 모델을 다시 학습합니다.'); return False
    if not MODEL_MANIFEST.is_file() or not (MODEL_DIR / 'training.csv').is_file(): print('저장된 모델이 없어 학습합니다.'); return False
    saved = json.loads(MODEL_MANIFEST.read_text(encoding='utf-8'))
    if saved['data']['csv_sha256'] != DATA_PROVENANCE['csv_sha256'] or saved['ids'] != IDS: print('데이터 또는 주차장 구성이 달라 다시 학습합니다.'); return False
    installed = importlib.metadata.version('scikit-learn')
    if saved['packages']['scikit-learn'] != installed: print(f"scikit-learn 버전이 달라(저장 {saved['packages']['scikit-learn']} / 현재 {installed}) 다시 학습합니다."); return False
    ok = all((MODEL_DIR / Path(m['path']).name).is_file() and file_sha256(MODEL_DIR / Path(m['path']).name) == m['sha256'] for m in saved['models'])
    if not ok: print('모델 파일 해시가 기록과 달라 다시 학습합니다.')
    return ok

training_rows = []
if saved_models_usable():
    training_rows = pd.read_csv(MODEL_DIR / 'training.csv').to_dict('records')
    print('저장된 모델 10개를 재사용합니다 (해시·환경 일치). 학습을 건너뜁니다.', flush=True)
else:
    uk = holidays.GB(years=sorted(data.LastUpdated.dt.year.unique().tolist()))
    for ident in IDS:
        g = data[data.SystemCodeNumber == ident].copy().sort_values('LastUpdated')
        g['minutes_of_day'] = g.LastUpdated.dt.hour * 60 + g.LastUpdated.dt.minute
        g['day_of_week'] = g.LastUpdated.dt.dayofweek
        g['day_type'] = [int(d.weekday() < 5 and d.date() not in uk) for d in g.LastUpdated]
        X = g[['minutes_of_day', 'day_of_week', 'day_type']]; y = g.Capacity - g.Occupancy
        split = int(len(g) * .8)
        model = RandomForestRegressor(n_estimators=1000, random_state=42, n_jobs=1)
        model.fit(X.iloc[:split], y.iloc[:split])
        mae = float(mean_absolute_error(y.iloc[split:], model.predict(X.iloc[split:])))
        joblib.dump(model, MODEL_DIR / f'parking_model_{ident}.pkl')
        training_rows.append(dict(id=ident, rows=len(g), train_rows=split, test_rows=len(g) - split, mae=mae, capacity=CAPACITY[ident], start=str(g.LastUpdated.min()), end=str(g.LastUpdated.max())))
        print('TRAINED', ident, 'rows', len(g), 'trees', model.n_estimators, 'holdout MAE', round(mae, 4), flush=True)
    pd.DataFrame(training_rows).to_csv(MODEL_DIR / 'training.csv', index=False)
    MODEL_MANIFEST.write_text(json.dumps({
        'requirement': 'SNR-PER-016', 'ids': IDS,
        'training': {'algorithm': 'RandomForestRegressor', 'n_estimators': 1000, 'random_state': 42, 'n_jobs': 1,
                     'features': ['minutes_of_day', 'day_of_week', 'day_type'], 'target': 'Capacity - Occupancy', 'split': 'chronological 80/20'},
        'data': {k: DATA_PROVENANCE[k] for k in ['url', 'doi', 'license', 'archive_sha256', 'csv_sha256', 'original_rows', 'clean_rows', 'start', 'end']},
        'python': sys.version, 'platform': platform.platform(),
        'packages': {p: importlib.metadata.version(p) for p in required.values()},
        'models': [{'id': i, 'path': f'models/parking_model_{i}.pkl', 'bytes': (MODEL_DIR / f'parking_model_{i}.pkl').stat().st_size,
                    'sha256': file_sha256(MODEL_DIR / f'parking_model_{i}.pkl')} for i in IDS]}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('모델 10개와 MODEL_MANIFEST.json을 저장했습니다.', flush=True)
display(pd.DataFrame(training_rows))

## 5. 시험 실행 폴더와 시험 환경 기록

모델 파일 10개가 모두 있는지 확인하고, `results/<실행 시각>/` 폴더를 만듭니다. 예측 기준 시각(`ANCHOR`)은 **데이터 마지막 시각의 다음 정시**이며 이후 1시간 간격 48개 시각(`DATES`)을 예측합니다.

`manifest.json`에는 시험 시각 기준, 모델 파일 크기·SHA-256, Python·패키지 버전, 컨테이너 CPU·메모리 제한(cgroup)을 기록합니다. `data-provenance.json`, `training.csv`도 함께 저장합니다.

In [ ]:
assert REPEATS >= 1 and BASELINE_REPEATS >= 1
missing = [str(MODEL_DIR / f'parking_model_{i}.pkl') for i in IDS if not (MODEL_DIR / f'parking_model_{i}.pkl').is_file()]
if missing:
    raise FileNotFoundError('모델 파일이 없습니다:\n' + '\n'.join(missing))
RUN_DIR = OUTPUT_ROOT / datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'responses').mkdir()
(RUN_DIR / 'figures').mkdir()
ANCHOR = (data.LastUpdated.max().to_pydatetime() + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0, tzinfo=ZoneInfo('Europe/London'))
DATES = [ANCHOR + timedelta(hours=i) for i in range(48)]
manifest = {'requirement': 'SNR-PER-016', 'anchor': ANCHOR.isoformat(), 'models': [],
            'python': sys.version, 'platform': platform.platform(),
            'packages': {p: importlib.metadata.version(p) for p in required.values()}}
for ident in IDS:
    p = MODEL_DIR / f'parking_model_{ident}.pkl'
    manifest['models'].append({'id': ident, 'path': p.relative_to(PACKAGE).as_posix(), 'bytes': p.stat().st_size,
                               'sha256': hashlib.sha256(p.read_bytes()).hexdigest()})
for filename in ['cpu.max', 'memory.max']:
    p = Path('/sys/fs/cgroup') / filename
    manifest[filename] = p.read_text().strip() if p.exists() else 'unavailable'
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(manifest['models']))
print('출력 경로:', RUN_DIR)
(RUN_DIR / 'data-provenance.json').write_text(json.dumps(DATA_PROVENANCE, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(training_rows).to_csv(RUN_DIR / 'training.csv', index=False)

## 6. 예측 API 서버 (Flask)

`GET /api/tci/ai_engine/park_pred?measure_position_id=<ID>&mode=batch|sequential`

- 요청마다 **모델 파일을 새로 읽습니다**(`model_loading: per-request`). 서버 시작 시간은 측정에 포함하지 않습니다.
- `mode=batch`(기본): 48개 시각의 특징을 표 하나로 만들어 **한 번에** 예측합니다. 공휴일 달력은 서버가 한 번 만들어 재사용합니다.
- `mode=sequential`: 이전 방식과 같이 시각마다 한 건씩 **48번** 예측하고 호출마다 달력을 새로 만듭니다. 예측값 일치와 개선 효과를 비교하는 기준입니다.
- 응답: `date_time`, `prediction_value`(정수), `raw_prediction_value`, `congestion_rate`와 서버 측정 시간 `inference_duration_seconds`.
- 잘못된 주소는 404, 잘못된 `measure_position_id`·`mode`는 400, 예측 실패는 500을 돌려줍니다.

In [ ]:
app = Flask('snr016_notebook')
calendar = holidays.GB(years=sorted({d.year for d in DATES}))
def frame(dates, cal):
    return pd.DataFrame([[d.hour*60+d.minute, d.weekday(), int(d.weekday()<5 and d.date() not in cal)]
                         for d in dates], columns=['minutes_of_day', 'day_of_week', 'day_type'])
@app.get('/')
def health(): return jsonify(status='ready', requirement='SNR-PER-016')
@app.get('/api/tci/ai_engine/park_pred')
def predict():
    started = perf_counter_ns()
    raw = request.args.get('measure_position_id', '')
    mode = request.args.get('mode', 'batch')
    if raw not in IDS or mode not in ['batch', 'sequential']:
        return jsonify(success=False, error='Invalid measure_position_id or mode'), 400
    ident = raw
    try:
        model = joblib.load(MODEL_DIR / f'parking_model_{ident}.pkl')
        if mode == 'batch':
            values = model.predict(frame(DATES, calendar))
        else:
            values = np.array([model.predict(frame([d], holidays.GB(years=d.year)))[0] for d in DATES])
        if len(values) != 48 or not np.isfinite(values).all(): raise ValueError('Invalid predictions')
        predictions = [{'date_time': d.isoformat(), 'prediction_value': int(v),
                        'raw_prediction_value': float(v),
                        'congestion_rate': round(max(0, min(1, 1-float(v)/CAPACITY[ident])), 2)}
                       for d, v in zip(DATES, values)]
        result = {'success': True, 'measure_position_id': ident, 'mode': mode,
                  'model_loading': 'per-request', 'predictions': predictions}
        result['inference_duration_seconds'] = (perf_counter_ns()-started)/1e9
        return jsonify(result), 200
    except Exception as exc:
        return jsonify(success=False, error=f'{type(exc).__name__}: {exc}'), 500
@app.errorhandler(404)
def not_found(exc): return jsonify(success=False, error='Invalid URL'), 404
class QuietHandler(WSGIRequestHandler):
    def log_request(self, *args, **kwargs): pass

## 7. 반복 측정

서버를 같은 컨테이너의 `127.0.0.1` 임의 포트에 띄우고 `requests`로 호출합니다(프록시 환경변수는 사용하지 않음).

1. **기준 측정:** `sequential` 모드로 주차장마다 `BASELINE_REPEATS`회. 이 예측값이 비교 기준입니다.
2. **본 측정:** `batch` 모드로 주차장마다 `REPEATS`회(10곳 × 30회 = **300회**). **첫 요청도 제외하지 않고** 모두 집계합니다.
3. 응답마다 검사: HTTP 200, `success`, 주차장 ID, 예측값 48개, 시각 간격 3,600초, 값이 유한한 수인지, 서버·클라이언트 시간이 2초 이하인지.
4. `batch`와 `sequential`의 `raw_prediction_value`가 `rtol=1e-10, atol=1e-8` 안에서 같은지 확인합니다.
5. 오류 응답 3건을 시험합니다: `/invalid`→404, 잘못된 ID→400, ID 없음→400.

응답 원문은 `responses/`, 측정표는 `measurements.csv`·`measurements.jsonl`에 저장합니다.

In [ ]:
rows = []; errors = []; equivalence = []; reference = {}; examples = {}
def sample(session, base, ident, mode, repeat):
    start = perf_counter_ns()
    r = session.get(base+'/api/tci/ai_engine/park_pred', params={'measure_position_id': ident, 'mode': mode}, timeout=60)
    payload = r.content
    client_seconds = (perf_counter_ns()-start)/1e9
    data = r.json(); pred = data.get('predictions', [])
    dates = [datetime.fromisoformat(v['date_time']) for v in pred]
    valid = (r.status_code == 200 and data.get('success') is True and data.get('measure_position_id') == ident
             and len(pred) == 48 and all((b-a).total_seconds() == 3600 for a, b in zip(dates, dates[1:]))
             and all(math.isfinite(v['raw_prediction_value']) for v in pred))
    duration = data.get('inference_duration_seconds')
    server_ok = valid and isinstance(duration, (int, float)) and 0 <= duration <= THRESHOLD_SECONDS
    client_ok = valid and 0 <= client_seconds <= THRESHOLD_SECONDS
    record = {'mode': mode, 'repeat': repeat, 'measure_position_id': ident, 'http_status': r.status_code,
              'count': len(pred), 'response_valid': valid, 'server_seconds': duration, 'client_seconds': client_seconds,
              'server_pass': server_ok, 'client_pass': client_ok, 'response_sha256': hashlib.sha256(payload).hexdigest()}
    (RUN_DIR/'responses'/f'{mode}_{repeat:02d}_{ident}.json').write_bytes(payload)
    rows.append(record)
    with (RUN_DIR/'measurements.jsonl').open('a', encoding='utf-8') as f: f.write(json.dumps(record)+'\n')
    if not valid: raise RuntimeError(f'응답 검증 실패: {ident}, HTTP {r.status_code}: {data}')
    values = np.array([v['raw_prediction_value'] for v in pred])
    if mode == 'sequential': reference[ident] = values
    else:
        equal = bool(np.allclose(values, reference[ident], rtol=1e-10, atol=1e-8))
        equivalence.append({'id': ident, 'repeat': repeat, 'equal': equal,
                            'max_absolute_difference': float(np.max(np.abs(values-reference[ident])))})
        examples[ident] = pred
    return record
server = make_server('127.0.0.1', 0, app, threaded=False, request_handler=QuietHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start(); base = f'http://127.0.0.1:{server.server_port}'
try:
    with requests.Session() as session:
        session.trust_env = False
        assert session.get(base+'/', timeout=10).json()['status'] == 'ready'
        for mode, repeats in [('sequential', BASELINE_REPEATS), ('batch', REPEATS)]:
            for repeat in range(1, repeats+1):
                for ident in IDS:
                    row = sample(session, base, ident, mode, repeat)
                    print(f"{mode} {repeat}/{repeats} ID={ident} server={row['server_seconds']:.6f}s client={row['client_seconds']:.6f}s", flush=True)
        for path, status in [('/invalid', 404), ('/api/tci/ai_engine/park_pred?measure_position_id=999', 400), ('/api/tci/ai_engine/park_pred', 400)]:
            r = session.get(base+path, timeout=10)
            errors.append({'path': path, 'expected': status, 'actual': r.status_code, 'passed': r.status_code == status, 'response': r.json()})
finally:
    server.shutdown(); server.server_close(); thread.join(timeout=10)
    pd.DataFrame(rows).to_csv(RUN_DIR/'measurements.csv', index=False, encoding='utf-8-sig')
print('서버 종료 및 원자료 저장 완료')

## 8. 판정 요약

본 측정 300회의 서버 시간·클라이언트 시간이 모두 2초 이하인지, 예측값 일치, 오류 응답 처리를 종합해 `overall_pass`를 정합니다. 주차장별 평균·최대·95백분위와 기준 측정 대비 개선 배율을 `per_id_summary.csv`에 저장합니다.

In [ ]:
df = pd.DataFrame(rows); batch = df[df['mode'] == 'batch']; sequential = df[df['mode'] == 'sequential']
same = all(x['equal'] for x in equivalence)
summary = {'requirement': 'SNR-PER-016', 'threshold_seconds': THRESHOLD_SECONDS,
           'requests': len(batch), 'server_pass': bool(batch['server_pass'].all()),
           'client_pass': bool(batch['client_pass'].all()), 'equivalence_pass': same,
           'error_handling_pass': all(x['passed'] for x in errors),
           'max_server_seconds': float(batch['server_seconds'].max()),
           'p95_server_seconds': float(batch['server_seconds'].quantile(.95)),
           'max_client_seconds': float(batch['client_seconds'].max()),
           'server_over_2_seconds': int((batch['server_seconds'] > THRESHOLD_SECONDS).sum()),
           'client_over_2_seconds': int((batch['client_seconds'] > THRESHOLD_SECONDS).sum()),
           'measurement_scope': 'single container loopback HTTP; models reloaded each request; startup excluded; first request retained',
           'baseline_note': 'same model/input, original-style 48 calls and per-call calendar construction; baseline repeat count may differ'}
summary['overall_pass'] = all(summary[x] for x in ['server_pass', 'client_pass', 'equivalence_pass', 'error_handling_pass'])
for name, value in [('summary', summary), ('errors', errors), ('equivalence', equivalence)]:
    (RUN_DIR/f'{name}.json').write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
stats = batch.groupby('measure_position_id').agg(count=('server_seconds', 'count'), mean_seconds=('server_seconds', 'mean'),
    max_seconds=('server_seconds', 'max'), client_max_seconds=('client_seconds', 'max'))
stats['p95_seconds'] = batch.groupby('measure_position_id')['server_seconds'].quantile(.95)
stats['baseline_mean_seconds'] = sequential.groupby('measure_position_id')['server_seconds'].mean()
stats['speedup_ratio'] = stats['baseline_mean_seconds']/stats['mean_seconds']
stats.to_csv(RUN_DIR/'per_id_summary.csv', encoding='utf-8-sig')
display(stats)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 9. 그래프

`figures/`에 PNG 9개를 저장합니다: 개선 전후 비교, 주차장별 최대·95백분위, 처리시간 분포, 요청별 지연, 48시간 예측 곡선, 순차·일괄 예측 일치, 측정 구간, 데이터 수, 홀드아웃 MAE.

In [ ]:
plt.style.use('default')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.size': 10})
def savefig(name):
    plt.tight_layout(); plt.savefig(RUN_DIR/'figures'/name, dpi=170, bbox_inches='tight'); plt.show(); plt.close()
x = np.arange(len(IDS)); labels = [str(i) for i in IDS]
plt.figure(figsize=(12, 4)); plt.bar(x-.18, stats.loc[IDS, 'baseline_mean_seconds'], .36, label='48 separate calls'); plt.bar(x+.18, stats.loc[IDS, 'mean_seconds'], .36, label='48-row batch')
plt.axhline(2, color='red', linestyle='--', label='2 s target'); plt.xticks(x, labels, rotation=35); plt.ylabel('Seconds'); plt.title('Same-model before and after comparison'); plt.legend(); savefig('01-before-after.png')
plt.figure(figsize=(12, 4)); plt.bar(x, stats.loc[IDS, 'max_seconds'], label='Maximum'); plt.scatter(x, stats.loc[IDS, 'p95_seconds'], color='orange', label='95th percentile'); plt.axhline(2, color='red', linestyle='--', label='2 s target'); plt.xticks(x, labels, rotation=35); plt.ylabel('Seconds'); plt.title('Batch inference by parking lot'); plt.legend(); savefig('02-per-id-max-p95.png')
plt.figure(figsize=(12, 4)); plt.boxplot([batch[batch.measure_position_id == i].server_seconds for i in IDS], tick_labels=labels); plt.xticks(rotation=35); plt.axhline(2, color='red', linestyle='--'); plt.ylabel('Seconds'); plt.title('All measured requests including first request'); savefig('03-distribution.png')
plt.figure(figsize=(11, 4)); plt.plot(batch.server_seconds.to_numpy(), label='Server'); plt.plot(batch.client_seconds.to_numpy(), label='Complete HTTP response', alpha=.7); plt.axhline(2, color='red', linestyle='--'); plt.xlabel('Request index'); plt.ylabel('Seconds'); plt.title('Repeated request latency'); plt.legend(); savefig('04-request-latency.png')
fig, axes = plt.subplots(5, 2, figsize=(13, 14))
for ax, ident in zip(axes.flat, IDS):
    ax.plot(range(48), [p['raw_prediction_value'] for p in examples[ident]], color='#2563eb'); ax.set_title(str(ident)); ax.set_xlabel('Forecast horizon (hours)'); ax.set_ylabel('Available spaces'); ax.grid(alpha=.2)
savefig('05-48-hour-predictions.png')
eq = pd.DataFrame(equivalence).groupby('id')['max_absolute_difference'].max()
plt.figure(figsize=(12, 4)); plt.bar(x, eq.loc[IDS]); plt.xticks(x, labels, rotation=35); plt.ylabel('Absolute difference'); plt.title('Sequential vs batch prediction equivalence'); savefig('06-equivalence.png')
fig, ax = plt.subplots(figsize=(12, 3)); ax.axis('off')
for xpos, label in [(.02, 'Request received'), (.27, 'Load real model'), (.52, 'Features + prediction'), (.79, 'JSON response')]:
    ax.text(xpos, .65, label, transform=ax.transAxes, bbox=dict(boxstyle='round', fc='#eff6ff', ec='#2563eb'))
ax.annotate('', xy=(.96, .3), xytext=(.02, .3), arrowprops=dict(arrowstyle='<->', color='#2563eb')); ax.text(.2, .1, 'Client timing includes complete HTTP response; server timing ends before serialization', transform=ax.transAxes)
savefig('07-measurement-scope.png')
plt.figure(figsize=(12, 4)); plt.bar(range(10), [x['rows'] for x in training_rows]); plt.xticks(range(10), IDS, rotation=45); plt.ylabel('Records'); plt.title('Observed records per parking lot'); savefig('08-data-coverage.png')
plt.figure(figsize=(12, 4)); plt.bar(range(10), [x['mae'] for x in training_rows]); plt.xticks(range(10), IDS, rotation=45); plt.ylabel('Available spaces MAE'); plt.title('Chronological 20 percent holdout'); savefig('09-holdout-mae.png')

## 10. 반복 전처리 재사용 시험 (공휴일 정보 재사용)

공휴일 달력 객체를 요청 사이에 **재사용**하면서, 48개 시각을 시각마다 한 건씩 **개별 추론**하는 구성을 같은 모델·같은 입력으로 시험합니다.

- 주차장별 30회, 총 300회, 요청마다 모델 파일을 새로 읽습니다.
- 검증: 응답 구조, 시각 간격, 7번 단계의 `sequential` 기준 예측값과 일치 여부, 평균·최대·95백분위 시간, 서버·클라이언트 시간이 모두 2초 이하인지.
- 결과는 `calendar_cached/`에 저장하고 그래프 `10-calendar-cached-latency.png`를 추가합니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# 동일 1000-tree 모델, 48개 시점 개별 추론 및 휴일 정보 재사용
cached_app = Flask('calendar_cached_profile')
@cached_app.get('/api/tci/ai_engine/park_pred')
def cached_predict():
    start = perf_counter_ns(); ident = request.args.get('measure_position_id', '')
    if ident not in IDS: return jsonify(success=False, error='Invalid measure_position_id'), 400
    model = joblib.load(MODEL_DIR/f'parking_model_{ident}.pkl')
    values = [float(model.predict(frame([d], calendar))[0]) for d in DATES]
    predictions = [dict(date_time=d.isoformat(), raw_prediction_value=v, prediction_value=int(v)) for d, v in zip(DATES, values)]
    result = dict(success=True, measure_position_id=ident, predictions=predictions, mode='calendar_cached', model_loading='per-request')
    result['inference_duration_seconds'] = (perf_counter_ns()-start)/1e9
    return jsonify(result)
@cached_app.errorhandler(404)
def cached_404(exc): return jsonify(success=False, error='Invalid URL'), 404
profile_dir = RUN_DIR/'calendar_cached'; profile_dir.mkdir(exist_ok=True)
(profile_dir/'responses').mkdir(exist_ok=True)
profile_rows = []
profile_server = make_server('127.0.0.1', 0, cached_app, threaded=False, request_handler=QuietHandler)
profile_thread = threading.Thread(target=profile_server.serve_forever, daemon=True); profile_thread.start()
try:
    with requests.Session() as session:
        session.trust_env = False
        for repeat in range(1, 31):
            for ident in IDS:
                start = perf_counter_ns()
                response = session.get(f'http://127.0.0.1:{profile_server.server_port}/api/tci/ai_engine/park_pred', params={'measure_position_id': ident}, timeout=30)
                raw = response.content; elapsed = (perf_counter_ns()-start)/1e9; obj = response.json(); pred = obj.get('predictions', [])
                equivalent = len(pred) == 48 and np.allclose([v['raw_prediction_value'] for v in pred], reference[ident], rtol=1e-10, atol=1e-8)
                dates = [datetime.fromisoformat(v['date_time']) for v in pred]
                valid = response.status_code == 200 and obj.get('success') is True and equivalent and all((b-a).total_seconds() == 3600 for a, b in zip(dates, dates[1:]))
                row = dict(repeat=repeat, id=ident, server_seconds=obj.get('inference_duration_seconds'), client_seconds=elapsed, valid=bool(valid), equivalent=bool(equivalent))
                profile_rows.append(row)
                (profile_dir/'responses'/f'{repeat:02d}_{ident}.json').write_bytes(raw)
                with (profile_dir/'measurements.jsonl').open('a') as f: f.write(json.dumps(row)+'\n')
            print('CACHED_PROFILE', repeat, '/30', 'mean_seconds', round(np.mean([x['server_seconds'] for x in profile_rows]), 6), flush=True)
finally:
    profile_server.shutdown(); profile_server.server_close(); profile_thread.join(timeout=10)
    pd.DataFrame(profile_rows).to_csv(profile_dir/'measurements.csv', index=False)
profile_df = pd.DataFrame(profile_rows)
profile_summary = dict(profile='calendar_cached', requests=len(profile_df), mean_server_seconds=float(profile_df.server_seconds.mean()), max_server_seconds=float(profile_df.server_seconds.max()), p95_server_seconds=float(profile_df.server_seconds.quantile(.95)), mean_client_seconds=float(profile_df.client_seconds.mean()), max_client_seconds=float(profile_df.client_seconds.max()), over_2_seconds=int((profile_df.server_seconds > 2).sum()), valid=bool(profile_df.valid.all()), all_server_within_2=bool((profile_df.server_seconds <= 2).all()), all_client_within_2=bool((profile_df.client_seconds <= 2).all()))
(profile_dir/'summary.json').write_text(json.dumps(profile_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(profile_summary, ensure_ascii=False, indent=2))
display(profile_df.groupby('id').server_seconds.agg(['count', 'mean', 'max']))
plt.figure(figsize=(12, 4)); plt.plot(profile_df.server_seconds.to_numpy()); plt.axhline(2, color='red', linestyle='--', label='2s requirement'); plt.axhline(profile_df.server_seconds.mean(), color='green', label='Measured mean'); plt.ylabel('Seconds'); plt.xlabel('Request index'); plt.title('48-step inference with reusable calendar'); plt.legend(); savefig('10-calendar-cached-latency.png')

## 11. 결과 저장

`report.html`(판정·요약표·그래프), `checksums.json`(결과 폴더 안 모든 파일의 SHA-256), 전체 묶음 `results/<실행 시각>.zip`(결과와 내려받은 데이터 포함)을 만듭니다. 결과 폴더의 파일 구성은 다음과 같습니다.

| 파일 | 내용 |
|---|---|
| `summary.json`, `per_id_summary.csv` | 판정 요약, 주차장별 통계 |
| `measurements.csv`, `measurements.jsonl`, `responses/` | 요청별 측정값과 응답 원문 |
| `calendar_cached/` | 공휴일 정보 재사용 시험(300회)의 측정값·요약·응답 원문 |
| `errors.json`, `equivalence.json` | 오류 응답 시험, 예측값 일치 시험 |
| `manifest.json`, `data-provenance.json`, `training.csv` | 모델 해시·시험 환경, 데이터 출처, 학습 결과 |
| `figures/`, `report.html`, `checksums.json` | 그래프, HTML 결과, 파일 해시 |

In [ ]:
from html import escape
figures = sorted((RUN_DIR/'figures').glob('*.png'))
page = '<html><head><meta charset="utf-8"><title>SNR-PER-016</title></head><body style="font-family:sans-serif;background:white;color:#172033;max-width:1200px;margin:40px auto">'
page += '<h1>SNR-PER-016 시험 결과</h1><p>판정: '+('PASS' if summary['overall_pass'] else 'FAIL')+'</p>'
page += '<p>4초에서 2초로 변경한 기준은 요구사항의 3차년도 및 5차년도 목표 차이이다. 실제 개선 효과는 동일 모델·입력 비교 측정으로 확인한다.</p>'
page += '<pre>'+escape(json.dumps(summary, ensure_ascii=False, indent=2))+'</pre>'+stats.to_html()
page += '<h2>공휴일 정보 재사용 시험</h2><pre>'+escape(json.dumps(profile_summary, ensure_ascii=False, indent=2))+'</pre>'
page += '<p>각 측정 원문은 responses 폴더, 파일 무결성은 manifest.json과 checksums.json에 기록한다. 본 결과는 같은 컨테이너 내 HTTP 측정이다.</p>'
for f in figures: page += '<h2>'+escape(f.stem)+'</h2><img style="max-width:100%" src="figures/'+f.name+'">'
page += '</body></html>'
(RUN_DIR/'report.html').write_text(page, encoding='utf-8')
checksums = {str(p.relative_to(RUN_DIR)): hashlib.sha256(p.read_bytes()).hexdigest() for p in RUN_DIR.rglob('*') if p.is_file()}
(RUN_DIR/'checksums.json').write_text(json.dumps(checksums, indent=2), encoding='utf-8')
bundle = RUN_DIR.with_suffix('.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob('*'):
        if p.is_file(): z.write(p, 'results/'+str(p.relative_to(RUN_DIR)))
    for p in DATA_DIR.glob('*'):
        if p.is_file(): z.write(p, 'data/'+p.name)
print('완료:', 'PASS' if summary['overall_pass'] else 'FAIL')
print('결과 묶음:', bundle)
import os
from urllib.parse import quote
prefix = os.environ.get('NB_PREFIX', '').rstrip('/')
for target, label in [(RUN_DIR/'report.html', '시각화 결과 열기'), (bundle, '전체 시험 결과 ZIP 다운로드')]:
    try:
        relative = target.relative_to(Path.home()).as_posix()
    except ValueError:
        continue
    display(HTML('<a target="_blank" href="'+prefix+'/files/'+quote(relative)+'">'+label+'</a>'))